# Setup

In [ ]:
from matplotlib.pyplot import xlabel

from autotest_exp import load_final_results, get_final_analysis_df, get_columns_num
import pandas as pd

In [ ]:
results_dict = load_final_results()

In [ ]:
analysis_df = get_final_analysis_df(results_dict)
analysis_df['io_e'] = analysis_df['integration_option'].astype(str) + '_' + analysis_df['execution'].astype(str)

In [ ]:
avg_metrics = (
    analysis_df.groupby(['dataset', 'integration_option', 'labels'], as_index=False)
    [['total_recall', 'total_precision', 'total_fscore']]
    .mean()
)

In [ ]:
avg_metrics = avg_metrics[avg_metrics['labels'] < 2586]

# Graph Metrics

In [ ]:
import matplotlib.cm as cm
to_plot = ["total_fscore", "total_precision", "total_recall"]

groups = ['0', '1', '3', '2']
print(groups)
cmap = cm.get_cmap('tab10', 10)
group_colors = {g: cmap(i) for i, g in enumerate(groups)}

for dataset in analysis_df['dataset'].unique():
    print(dataset)
    for i in range(len(to_plot)):
        print(to_plot[i])
        df = pd.DataFrame()
        avg_metrics_for_dataset = analysis_df[analysis_df['dataset'] == dataset]
        for integration_option in avg_metrics_for_dataset['io_e'].unique():
            series = avg_metrics_for_dataset[avg_metrics_for_dataset['io_e'] == integration_option].sort_values(by='labels', ignore_index=True)[['labels', to_plot[i]]].set_index('labels')
            df[integration_option] = series
        df['labels'] = df.index / get_columns_num(f"datasets/DGov_Typo_subsets/{dataset}")

        cols = [c for c in df.columns if c != 'labels']
        colors = [group_colors[str(col).split('_')[0]] for col in cols]

        df.plot('labels',
                # logx=True,
                y=cols,
                color=colors,
                xlabel = "labels per column",
                xticks = list(df['labels']),
                yticks = [i/10 for i in range(11)],
                kind = "line",
                title=to_plot[i],
                linewidth = 0.8)


In [ ]:
to_plot = ["total_fscore", "total_precision", "total_recall"]
for dataset in avg_metrics['dataset'].unique():
    print(dataset)
    for i in range(len(to_plot)):
        print(to_plot[i])
        df = pd.DataFrame()
        avg_metrics_for_dataset = avg_metrics[avg_metrics['dataset'] == dataset]
        for integration_option in avg_metrics_for_dataset['integration_option'].unique():
            series = avg_metrics_for_dataset[avg_metrics_for_dataset['integration_option'] == integration_option].sort_values(by='labels', ignore_index=True)[['labels', to_plot[i]]].set_index('labels')
            df[integration_option] = series
        df['labels'] = df.index
        df['labels'] = df['labels'] / get_columns_num(f"datasets/DGov_Typo_subsets/{dataset}")
        df.plot('labels',
                # logx=True,
                xticks = list(df['labels']),
                xlabel = "labels per column",
                yticks = [i/10 for i in range(11)],
                kind = "line",
                title=to_plot[i],
                rot=-70)
